# DuckDB and Modern SQL

Vaultline's transaction history is growing fast. A year of data is
already 100,000 rows, and that's just one table. The analytics team
wants monthly summaries, fraud detection patterns, and merchant
breakdowns. SQLite was fine when we had a few thousand rows, but
aggregating the full history is starting to feel slow.

There's a better tool for analytical queries. It's called DuckDB, and
it runs right here in your browser.

## Setup

In [ ]:
%pip install -q duckdb

In [ ]:
import duckdb
import time

## The file IS the table

This is the thing that surprises people the first time they see it.
DuckDB can query a Parquet file directly, with plain SQL. No
`CREATE TABLE`. No `COPY FROM`. No ETL. You just point SQL at a file.

In [ ]:
duckdb.sql("SELECT * FROM '../data/transactions_historical.parquet' LIMIT 10")

Read that query again. The file path is in the `FROM` clause, where
you'd normally put a table name. DuckDB treats the file as a table.
It reads the Parquet metadata, infers the schema, and gives you
SQL access to the data.

This is the lakehouse concept in one line: data stays in files,
the compute engine reads them directly. No separate database to
load data into. No synchronisation to worry about. The file is
the source of truth, and the SQL engine is just a lens over it.

## How many rows are we working with?

Let's get a sense of the dataset.

In [ ]:
duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(timestamp) AS earliest,
        MAX(timestamp) AS latest,
        COUNT(DISTINCT customer_id) AS unique_customers,
        COUNT(DISTINCT merchant_name) AS unique_merchants
    FROM '../data/transactions_historical.parquet'
""")

100,000 transactions spanning a full year. Let's see how DuckDB
handles some real analytical queries.

## Why DuckDB and not SQLite?

SQLite is excellent for transactional workloads: inserting a single
row, looking up a record by ID, updating a status. It stores data
in rows, which makes those operations fast.

But analytical queries are different. When you ask "what's the
average transaction amount by merchant category per month?", the
database needs to scan an entire column across many rows. SQLite
reads each row in full even if you only need one column.

DuckDB stores data in columns. When you query `amount`, it only reads
the amount column. It also uses vectorised execution, processing
batches of values at once rather than one row at a time.

| Feature | SQLite | DuckDB |
|---|---|---|
| Storage | Row-oriented | Column-oriented |
| Best for | OLTP (transactions) | OLAP (analytics) |
| Single row lookup | Fast | Acceptable |
| Full column scan | Slow on large data | Fast |
| Parquet support | None | Native |
| Execution model | Row-at-a-time | Vectorised |

Let's see the difference in practice.

## Timing an analytical query

Let's run a realistic analytical query: monthly revenue by merchant
category, for completed transactions only.

In [ ]:
start = time.time()

result = duckdb.sql("""
    SELECT
        STRFTIME(timestamp::TIMESTAMP, '%Y-%m') AS month,
        merchant_category,
        COUNT(*) AS transaction_count,
        ROUND(SUM(amount), 2) AS total_amount,
        ROUND(AVG(amount), 2) AS avg_amount
    FROM '../data/transactions_historical.parquet'
    WHERE status = 'completed'
    GROUP BY month, merchant_category
    ORDER BY month, total_amount DESC
""")

elapsed = time.time() - start
print(f"Query completed in {elapsed:.3f} seconds")
result

That's a GROUP BY across 100,000 rows with filtering, aggregation,
and sorting. DuckDB handles it comfortably.

The same query pattern on a million rows, or ten million, would still
be fast. DuckDB was built for this.

## Fraud detection patterns

The compliance team wants to flag suspicious activity. One common
pattern: customers with a high number of large transactions in a
short period.

In [ ]:
duckdb.sql("""
    SELECT
        customer_id,
        COUNT(*) AS large_transactions,
        ROUND(SUM(amount), 2) AS total_amount,
        ROUND(MAX(amount), 2) AS max_single,
        MIN(timestamp) AS first_seen,
        MAX(timestamp) AS last_seen
    FROM '../data/transactions_historical.parquet'
    WHERE amount > 1000
      AND status = 'completed'
    GROUP BY customer_id
    HAVING COUNT(*) >= 5
    ORDER BY total_amount DESC
    LIMIT 15
""")

These customers have five or more transactions over 1,000 each. Are
they wealthy individuals making large purchases? Or is something
else going on? The data alone can't tell us, but it can tell us
where to look.

Let's also check for unusual processing times, which might indicate
system issues or attempted fraud.

In [ ]:
duckdb.sql("""
    SELECT
        status,
        COUNT(*) AS count,
        ROUND(AVG(processing_time_ms), 0) AS avg_processing_ms,
        ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY processing_time_ms), 0) AS p95_ms,
        MAX(processing_time_ms) AS max_ms
    FROM '../data/transactions_historical.parquet'
    GROUP BY status
    ORDER BY count DESC
""")

Notice the `PERCENTILE_CONT` function. This is a window function
that calculates percentiles directly in SQL -- something SQLite
cannot do. DuckDB supports the full range of modern analytical
SQL functions.

## Merchant analytics

The product team wants a merchant leaderboard: which merchants
generate the most revenue, and what's their typical transaction
size?

In [ ]:
duckdb.sql("""
    SELECT
        merchant_name,
        merchant_category,
        COUNT(*) AS transactions,
        ROUND(SUM(amount), 2) AS total_revenue,
        ROUND(AVG(amount), 2) AS avg_transaction,
        ROUND(SUM(CASE WHEN status = 'declined' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS decline_rate_pct
    FROM '../data/transactions_historical.parquet'
    GROUP BY merchant_name, merchant_category
    ORDER BY total_revenue DESC
    LIMIT 15
""")

The decline rate column is interesting. A merchant with a high
decline rate might have technical issues, or might be associated
with fraudulent activity. Either way, it's worth investigating.

## DuckDB to pandas

DuckDB results are great for SQL exploration, but sometimes you want
to pull the data into pandas for further analysis or visualisation.
The `.df()` method converts a DuckDB result to a pandas DataFrame.

In [ ]:
monthly_totals = duckdb.sql("""
    SELECT
        STRFTIME(timestamp::TIMESTAMP, '%Y-%m') AS month,
        ROUND(SUM(amount), 2) AS total_amount,
        COUNT(*) AS transactions
    FROM '../data/transactions_historical.parquet'
    WHERE status = 'completed'
    GROUP BY month
    ORDER BY month
""").df()

print(type(monthly_totals))
monthly_totals

Now it's a regular pandas DataFrame. You could plot it, merge it
with other data, or export it to a report. DuckDB does the heavy
lifting (filtering, grouping, aggregating), and pandas handles
the last mile.

## Querying multiple files

DuckDB can also query multiple Parquet files at once using glob
patterns. This is useful when your data is partitioned across
multiple files (a common pattern in data lakes).

In [ ]:
# Query both payment versions at once
duckdb.sql("""
    SELECT
        filename,
        COUNT(*) AS rows,
        ROUND(SUM(amount), 2) AS total,
        ROUND(AVG(amount), 2) AS avg_amount
    FROM read_parquet('../data/payments_v*.parquet', filename=true)
    GROUP BY filename
""")

The `filename=true` parameter adds a column showing which file each
row came from. DuckDB handles the schema differences automatically
by taking the union of all columns and filling missing ones with
null -- the same strategy we implemented manually in the schema
evolution notebook.

## The lakehouse concept

What we've been doing in this notebook is, in miniature, the
lakehouse architecture that major organisations use at scale.

The traditional approach is:

1. Store raw data in files (a "data lake")
2. Load data into a database (a "data warehouse")
3. Query the database

Step 2 is expensive, slow, and introduces a synchronisation problem:
the database is always slightly behind the files.

The lakehouse approach removes step 2:

1. Store data in Parquet files (the lake)
2. Query the files directly with an analytical engine like DuckDB

The data never moves. There's no ETL into a separate database. The
compute engine (DuckDB, Spark, Trino, etc.) reads the files
directly. This is simpler, cheaper, and eliminates the
synchronisation problem entirely.

At Vaultline's scale, DuckDB is more than enough. At larger scale,
the same concept applies with distributed engines like Apache Spark
or Trino -- the principle is identical, just the scale changes.

## Exercises

### Exercise 1

Write a DuckDB query that finds the top 10 customers by total
spending (completed transactions only). Show the customer ID,
total amount, number of transactions, and their most common
merchant category.

**Hint:** For "most common merchant category" you can use
`MODE(merchant_category)` which returns the most frequent value.

### Exercise 2

Write a query that compares weekday vs weekend transaction patterns.
Show the day of week (Monday through Sunday), total transactions,
average amount, and total amount.

**Hint:** Use `DAYNAME(timestamp::TIMESTAMP)` to get the day name,
and `DAYOFWEEK(timestamp::TIMESTAMP)` to get a number for sorting
(where 0 = Sunday).

### Exercise 3

Write a query that identifies the "rush hour" -- the hour of the
day with the most transactions. Show all 24 hours with their
transaction count and average amount.

**Hint:** Use `HOUR(timestamp::TIMESTAMP)` to extract the hour.

### Exercise 4

Write a query that calculates the month-over-month growth rate for
completed transactions. Show each month, its total amount, and the
percentage change from the previous month.

**Hint:** Use the `LAG()` window function to get the previous
month's total:
```sql
LAG(total_amount) OVER (ORDER BY month) AS prev_month
```

### Exercise 5

Use DuckDB to query the `transactions_historical.parquet` file and
convert the result to a pandas DataFrame. Then create a line chart
showing monthly transaction volume (number of transactions) over
the year.

**Hint:** You'll need to `%pip install -q matplotlib` and use the
`.df()` method to convert the DuckDB result.

## Summary

- **DuckDB** is a columnar analytical database that runs in-process
  (no server needed) and queries Parquet files directly
- **The file is the table**: `SELECT * FROM 'file.parquet'` just works,
  with no loading step required
- **Column-oriented storage** makes analytical queries (aggregations,
  scans, GROUP BY) much faster than row-oriented databases like SQLite
- **Modern SQL functions** like `PERCENTILE_CONT`, `MODE`, and window
  functions (`LAG`, `LEAD`) are fully supported
- **`.df()`** converts DuckDB results to pandas DataFrames for further
  analysis or visualisation
- **The lakehouse concept**: keep data in files, query it directly
  with an analytical engine. No ETL into a separate database.

This is the modern data engineering stack: Parquet for storage,
DuckDB (or Spark, or Trino) for compute, and Pydantic for validation.
The data stays where it is, and the tools come to the data.